# Clase 10 · Notebook 01
# Q-Learning desde cero: un robot que aprende a llegar a la meta

**Introducción al Deep Learning — Módulo 4, Unidad 2: Aprendizaje por refuerzo**

Vamos a implementar **Q-Learning** con NumPy, sin librerías especiales. Un robot debe cruzar una rejilla
desde la salida hasta la meta, evitando las casillas de fuego, **aprendiendo por prueba y error**.

1. Definir el entorno (gridworld) y las recompensas.
2. Crear la tabla Q.
3. Entrenar con la ecuación de Q-Learning (exploración/explotación).
4. Ver la política aprendida.

> 💡 Ejecuta las celdas en orden con **Shift + Enter**.

## 1. El entorno: una rejilla con salida, meta y fuego

Rejilla 4×4. El robot (estado = casilla) puede moverse arriba, abajo, izquierda o derecha.
Recompensas: **+10** llegar a la meta, **-10** caer en el fuego, **-1** por cada paso (para que busque el camino corto).

In [ ]:
import numpy as np
np.random.seed(42)

FILAS, COLS = 4, 4
META = (3, 3)
FUEGO = [(1, 1), (2, 3)]
ACCIONES = ["arriba", "abajo", "izquierda", "derecha"]
MOV = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}

def paso(estado, accion):
    df, dc = MOV[accion]
    nf, nc = estado[0] + df, estado[1] + dc
    if not (0 <= nf < FILAS and 0 <= nc < COLS):   # fuera de la rejilla: se queda
        nf, nc = estado
    nuevo = (nf, nc)
    if nuevo == META:    return nuevo, 10, True
    if nuevo in FUEGO:   return nuevo, -10, True
    return nuevo, -1, False

mapa = np.full((FILAS, COLS), ".", dtype=object)
mapa[META] = "M"
for f in FUEGO: mapa[f] = "F"
mapa[0, 0] = "S"
print("Entorno (S=salida, M=meta, F=fuego):")
print(mapa)

## 2. La tabla Q

Una fila por estado (16 casillas) y una columna por acción (4). Empieza toda a cero: el robot no sabe nada.

In [ ]:
Q = np.zeros((FILAS, COLS, len(ACCIONES)))
print("Forma de la tabla Q:", Q.shape, " (filas, columnas, acciones)")

## 3. Entrenamiento (Q-Learning)

En cada paso aplicamos la ecuación:

`Q(s,a) ← Q(s,a) + α · [ r + γ · max Q(s',·) − Q(s,a) ]`

- **α** (learning rate) = 0.1 · **γ** (descuento) = 0.9
- **ε-greedy**: con probabilidad ε exploramos (acción al azar); si no, explotamos (mejor acción conocida). ε baja con el tiempo.

In [ ]:
alpha, gamma = 0.1, 0.9
epsilon, eps_min, eps_decay = 1.0, 0.05, 0.999
EPISODIOS = 1500

recompensas = []
for ep in range(EPISODIOS):
    estado, terminado, total = (0, 0), False, 0
    for _ in range(50):                      # límite de pasos por episodio
        if np.random.rand() < epsilon:
            a = np.random.randint(4)         # explorar
        else:
            a = int(np.argmax(Q[estado]))    # explotar
        nuevo, r, terminado = paso(estado, a)
        # --- actualización de Q-Learning ---
        Q[estado][a] += alpha * (r + gamma * np.max(Q[nuevo]) - Q[estado][a])
        estado, total = nuevo, total + r
        if terminado: break
    epsilon = max(eps_min, epsilon * eps_decay)
    recompensas.append(total)

print("Entrenamiento terminado.")
print("Recompensa media de los primeros 50 episodios: %.1f" % np.mean(recompensas[:50]))
print("Recompensa media de los últimos 50 episodios:  %.1f" % np.mean(recompensas[-50:]))

In [ ]:
import matplotlib.pyplot as plt
media = np.convolve(recompensas, np.ones(30)/30, mode="valid")
plt.figure(figsize=(7, 4))
plt.plot(media, color="#000F9F")
plt.title("Recompensa por episodio (media móvil)"); plt.xlabel("Episodio"); plt.ylabel("Recompensa")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

La recompensa sube y se estabiliza: el robot ha aprendido a llegar a la meta evitando el fuego.

## 4. La política aprendida

Para cada casilla, mostramos la mejor acción según la tabla Q (la flecha que el robot seguiría).

In [ ]:
flechas = {0: "↑", 1: "↓", 2: "←", 3: "→"}
politica = np.full((FILAS, COLS), "", dtype=object)
for f in range(FILAS):
    for c in range(COLS):
        if (f, c) == META:      politica[f, c] = "M"
        elif (f, c) in FUEGO:   politica[f, c] = "F"
        else:                   politica[f, c] = flechas[int(np.argmax(Q[f, c]))]
print("Política aprendida (flecha = mejor acción en cada casilla):\n")
for fila in politica:
    print("  ".join(fila))

Siguiendo las flechas desde la salida (0,0) el robot llega a la meta esquivando el fuego.

## Resumen

- **Q-Learning** aprende una **tabla Q** (estado × acción) solo con experiencia (s, a, s', r).
- La **ecuación de actualización** mezcla la recompensa inmediata con el mejor valor futuro (γ).
- **ε-greedy** equilibra **explorar** (probar) y **explotar** (usar lo aprendido).
- La política óptima sale de elegir, en cada estado, la acción con mayor valor Q.

➡️ En el **Notebook 02** sustituiremos la tabla por una red de neuronas: una **DQN**.